# JASSIM AI — Google Colab GPU Server
دفتر تشغيل تجريبي لخادم JASSIM AI مع Ollama وقاعدة بيانات SQLite محفوظة في Google Drive ونفق HTTPS مؤقت.

تنبيه: Colab قد ينهي الجلسة المجانية أو يغيّر توفر GPU. خلية Watchdog تعيد تشغيل العمليات المتعطلة ما دامت الجلسة حية، لكنها لا تمنع إيقاف Colab. لا تشارك رابط النفق أو رمز API.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, subprocess, time, json, secrets, requests, re, threading

PROJECT = Path('/content/JASSIM-AI-Android')
DATA_DIR = Path('/content/drive/MyDrive/JASSIM_AI/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
print('Persistent database directory:', DATA_DIR)

## 1. تنزيل ملفات الخادم

In [ ]:
GIT_REF = 'main'
GIT_REPO = 'https://github.com/stevennpllk-crypto/-JASSIM-AI-Android.git'

if not PROJECT.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', GIT_REF, GIT_REPO, str(PROJECT)],
        check=True
    )
else:
    # Preserve any local edits before switching to the published source.
    status = subprocess.run(
        ['git', '-C', str(PROJECT), 'status', '--porcelain'],
        check=True, text=True, capture_output=True
    ).stdout.strip()
    if status:
        print('Local changes detected; preserving them in a Git stash before updating.')
        subprocess.run(
            ['git', '-C', str(PROJECT), 'stash', 'push', '--include-untracked',
             '-m', 'JASSIM-Colab-auto-preserve-before-main-update'],
            check=True
        )

    fetch = subprocess.run(
        ['git', '-C', str(PROJECT), 'fetch', '--depth', '1', 'origin', GIT_REF],
        text=True, capture_output=True
    )
    if fetch.returncode != 0:
        print(fetch.stdout)
        print(fetch.stderr)
        raise RuntimeError('Could not fetch JASSIM main branch; existing project files were preserved.')

    checkout = subprocess.run(
        ['git', '-C', str(PROJECT), 'checkout', '-B', GIT_REF, 'FETCH_HEAD'],
        text=True, capture_output=True
    )
    if checkout.returncode != 0:
        print(checkout.stdout)
        print(checkout.stderr)
        print('Preserved local changes remain in Git stash; inspect with: git -C /content/JASSIM-AI-Android stash list')
        raise RuntimeError('Could not switch JASSIM source to main; see Git output above.')

print('Using JASSIM source ref:', GIT_REF)
SERVER = PROJECT / 'backend' / 'server.py'
assert SERVER.exists(), 'backend/server.py is missing'
print('Server source ready:', SERVER)


## 2. إعداد الحماية والنموذج

In [ ]:
TOKEN_FILE = DATA_DIR / '.api_token'
if TOKEN_FILE.exists():
    API_TOKEN = TOKEN_FILE.read_text().strip()
else:
    API_TOKEN = secrets.token_urlsafe(48)
    TOKEN_FILE.write_text(API_TOKEN)
    try:
        TOKEN_FILE.chmod(0o600)
    except Exception:
        pass
print('Persistent private API token loaded from Google Drive; it is never printed automatically.')

os.environ['JASSIM_API_TOKEN'] = API_TOKEN
os.environ['JASSIM_CORS_ORIGIN'] = 'https://stevennpllk-crypto.github.io'
os.environ['JASSIM_HOST'] = '127.0.0.1'
os.environ['JASSIM_PORT'] = '8787'
os.environ['JASSIM_DATA_DIR'] = str(DATA_DIR)
os.environ['JASSIM_OLLAMA_URL'] = 'http://127.0.0.1:11434'
gpu_probe = subprocess.run(
    ['nvidia-smi', '--query-gpu=memory.total,memory.free', '--format=csv,noheader,nounits'],
    capture_output=True, text=True
)
gpu_memory_mib = 0
gpu_free_mib = 0
if gpu_probe.returncode == 0:
    for row in gpu_probe.stdout.splitlines():
        try:
            total_mib, free_mib = (int(value.strip()) for value in row.split(','))
            if free_mib > gpu_free_mib:
                gpu_memory_mib, gpu_free_mib = total_mib, free_mib
        except (ValueError, TypeError):
            continue
has_nvidia_gpu = gpu_memory_mib > 0

def choose_default_model(gpu_memory_mib, gpu_free_mib=None, override=None):
    """Select 9B only when a large GPU also has enough currently free VRAM."""
    if override and str(override).strip():
        return str(override).strip()
    enough_total = int(gpu_memory_mib) >= 14000
    enough_free = gpu_free_mib is not None and int(gpu_free_mib) >= 10000
    return 'qwen3.5:9b' if enough_total and enough_free else 'qwen3.5:4b'

default_model = choose_default_model(
    gpu_memory_mib, gpu_free_mib, os.environ.get('JASSIM_OLLAMA_MODEL')
)
os.environ['JASSIM_OLLAMA_MODEL'] = default_model
MODEL = os.environ['JASSIM_OLLAMA_MODEL']
print('NVIDIA GPU detected:', has_nvidia_gpu)
print('Detected GPU memory (MiB):', gpu_memory_mib)
print('Currently free GPU memory (MiB):', gpu_free_mib)
print('Selected model:', MODEL)
print('Automatic profile: qwen3.5:9b only with at least 14,000 MiB total and 10,000 MiB free VRAM; qwen3.5:4b otherwise. An explicit JASSIM_OLLAMA_MODEL override is preserved.')

## 3. تثبيت وتشغيل Ollama وتنزيل النموذج

In [ ]:
zstd_check = subprocess.run(['bash', '-lc', 'command -v zstd'], capture_output=True, text=True)
if zstd_check.returncode != 0:
    print('Installing the missing zstd extractor...')
    apt_update = subprocess.run(['apt-get', 'update', '-qq'], text=True, capture_output=True)
    if apt_update.returncode != 0:
        print(apt_update.stderr[-2000:])
        raise RuntimeError('Could not update apt package indexes before installing zstd')
    zstd_install = subprocess.run(['apt-get', 'install', '-y', '-qq', 'zstd'], text=True, capture_output=True)
    if zstd_install.returncode != 0:
        print(zstd_install.stdout[-2000:])
        print(zstd_install.stderr[-2000:])
        raise RuntimeError('Could not install zstd; Ollama installation was not attempted')
    subprocess.run(['bash', '-lc', 'zstd --version'], check=True, text=True)
else:
    print('zstd is already installed.')

ollama_proc = None
ollama_log = None
try:
    existing = requests.get('http://127.0.0.1:11434/api/tags', timeout=3)
    existing.raise_for_status()
    print('Reusing the already-running Ollama service.')
except Exception:
    install = subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, text=True, capture_output=True)
    print(install.stdout[-3000:])
    if install.returncode != 0:
        print(install.stderr[-3000:])
        raise RuntimeError('Ollama installation failed')
    ollama_log = open('/content/ollama.log', 'a')
    ollama_proc = subprocess.Popen(['ollama', 'serve'], stdout=ollama_log, stderr=subprocess.STDOUT)
    for _ in range(45):
        try:
            requests.get('http://127.0.0.1:11434/api/tags', timeout=3).raise_for_status()
            break
        except Exception:
            time.sleep(2)
    else:
        raise RuntimeError('Ollama did not start. Check /content/ollama.log')

pull = subprocess.run(['ollama', 'pull', MODEL], text=True)
if pull.returncode != 0:
    raise RuntimeError('Model download failed: ' + MODEL)
print('Ollama and model are ready; keep_alive=-1 keeps the model loaded while Ollama runs.')

## 4. تشغيل API مع تخزين دائم للمحادثات

In [ ]:
# Restart only an existing JASSIM server process launched from this exact project path.
import signal
server_path = str(SERVER.resolve())
stopped_pids = []
for proc_dir in Path('/proc').glob('[0-9]*'):
    try:
        pid = int(proc_dir.name)
        if pid == os.getpid():
            continue
        command = (proc_dir / 'cmdline').read_bytes().replace(b'\\x00', b' ').decode(errors='ignore')
        if server_path in command and 'python' in command:
            os.kill(pid, signal.SIGTERM)
            stopped_pids.append(pid)
    except (FileNotFoundError, ProcessLookupError, PermissionError, ValueError):
        pass
if stopped_pids:
    print('Stopped previous JASSIM API process(es):', stopped_pids)
    time.sleep(2)

api_log = open('/content/jassim-api.log', 'a')
api_proc = subprocess.Popen(
    ['python', str(SERVER)],
    cwd=str(PROJECT / 'backend'),
    env=os.environ.copy(),
    stdout=api_log,
    stderr=subprocess.STDOUT
)

for _ in range(45):
    try:
        health = requests.get('http://127.0.0.1:8787/ready', timeout=3)
        if health.ok and health.json().get('ready'):
            break
    except Exception:
        pass
    time.sleep(2)
else:
    log_path = Path('/content/jassim-api.log')
    print(log_path.read_text(errors='ignore')[-4000:] if log_path.exists() else 'API log missing')
    raise RuntimeError('JASSIM API/model readiness failed')
print(json.dumps(health.json(), ensure_ascii=False, indent=2))

## 5. اختبار إجابة حقيقية من النموذج

In [ ]:
# Direct Ollama probe must validate generated text, not only the HTTP status.
direct_probe = requests.post(
    'http://127.0.0.1:11434/api/chat',
    json={
        'model': MODEL,
        'messages': [{'role': 'user', 'content': 'Reply with one short Arabic greeting.'}],
        'stream': False,
        'keep_alive': -1,
        'options': {'num_ctx': 2048, 'num_predict': 64}
    },
    timeout=180
)
print('Direct Ollama probe HTTP:', direct_probe.status_code)
try:
    direct_payload = direct_probe.json()
except ValueError:
    direct_payload = {'raw_response': direct_probe.text[:3000]}

if not direct_probe.ok:
    print('Ollama rejection details:', json.dumps(direct_payload, ensure_ascii=False)[:3000])
    direct_probe.raise_for_status()

direct_error = str(direct_payload.get('error') or '').strip()
direct_message = direct_payload.get('message')
direct_answer = str(direct_message.get('content') or '').strip() if isinstance(direct_message, dict) else ''
if direct_error:
    print('Ollama model error:', direct_error[:2000])
    raise RuntimeError('Ollama returned HTTP success but reported a model error: ' + direct_error[:500])
if not direct_answer:
    print('Ollama response did not contain message.content:')
    print(json.dumps(direct_payload, ensure_ascii=False, indent=2)[:3000])
    raise RuntimeError('Ollama returned HTTP success but no generated message.content was present.')
print('Direct Ollama generated non-empty text:', direct_answer[:300])

headers = {'Authorization': 'Bearer ' + API_TOKEN, 'Content-Type': 'application/json'}
test = requests.post(
    'http://127.0.0.1:8787/api/chat',
    headers=headers,
    json={'message': 'عرّف بنفسك واشرح كيف تساعد المستخدم بالعربية.', 'conversation_id': 'colab-check'},
    timeout=300
)
print('HTTP:', test.status_code)
try:
    result = test.json()
except Exception:
    result = {'raw_response': test.text[:2000]}
print(json.dumps(result, ensure_ascii=False, indent=2))

def diagnose_private_engine():
    print('\n========== JASSIM PRIVATE ENGINE DIAGNOSTICS ==========')
    checks = [
        ('Ollama process', ['bash', '-lc', "pgrep -af 'ollama serve' || true"]),
        ('Installed models', ['ollama', 'list']),
    ]
    for label, command in checks:
        print(f'\n--- {label} ---')
        try:
            check = subprocess.run(command, capture_output=True, text=True, timeout=20)
            print((check.stdout or '')[-4000:])
            if check.stderr:
                print('stderr:', check.stderr[-2000:])
            print('exit_code:', check.returncode)
        except Exception as exc:
            print(type(exc).__name__ + ':', str(exc))

    for label, url in [
        ('Ollama tags', 'http://127.0.0.1:11434/api/tags'),
        ('Ollama loaded models', 'http://127.0.0.1:11434/api/ps'),
        ('JASSIM readiness', 'http://127.0.0.1:8787/ready'),
        ('JASSIM liveness', 'http://127.0.0.1:8787/health'),
    ]:
        print(f'\n--- {label} ---')
        try:
            response = requests.get(url, timeout=8)
            print('HTTP:', response.status_code)
            try:
                payload = response.json()
                if label == 'Ollama tags':
                    print('models:', [m.get('name') for m in payload.get('models', [])])
                elif label == 'Ollama loaded models':
                    print('loaded:', [m.get('name') for m in payload.get('models', [])])
                else:
                    print(json.dumps(payload, ensure_ascii=False, indent=2))
            except Exception:
                print(response.text[:2000])
        except Exception as exc:
            print(type(exc).__name__ + ':', str(exc))

    for label, path in [
        ('Ollama log', Path('/content/ollama.log')),
        ('JASSIM API log', Path('/content/jassim-api.log')),
    ]:
        print(f'\n--- {label} (last 80 lines) ---')
        if path.exists():
            print('\n'.join(path.read_text(errors='ignore').splitlines()[-80:]))
        else:
            print('Log file not found:', path)

    print('\n========== END DIAGNOSTICS ==========')

if not test.ok:
    diagnose_private_engine()
    raise RuntimeError(
        'Private inference test failed. Review the diagnostics above; '
        'no external AI provider was contacted.'
    )

assert result.get('engine') == 'ollama', 'Response did not come from the private Ollama engine'
assert (result.get('answer') or result.get('reply') or '').strip(), 'Private model returned an empty answer'
print('SUCCESS: a real answer was generated by the private Ollama model.')


## 6. إنشاء رابط HTTPS مؤقت

بعد نجاح اختبار النموذج، تنشئ هذه الخلية رابط HTTPS مؤقتاً عبر Cloudflare Tunnel.

**الإعداد الافتراضي آمن:** `PUBLISH_ENDPOINT_TO_GITHUB = False`، لذلك لا تحتاج إلى إنشاء GitHub Token ولا تُنشر بيانات الاتصال تلقائياً. انسخ رابط HTTPS الذي يظهر بعد تشغيل الخلية، ثم أدخله يدوياً في إعدادات اتصال Android. سيظل رمز API منفصلاً ولا تضعه في رابط الخادم.

### تفعيل الاكتشاف التلقائي اختيارياً فقط
إذا أردت لاحقاً نشر عنوان النفق المؤقت إلى `dashboard/connection.json` ليكتشفه التطبيق تلقائياً، فعّل `PUBLISH_ENDPOINT_TO_GITHUB` بنفسك داخل خلية النفق، ثم أضف GitHub Fine-grained token في Colab Secrets باسم `JASSIM_GITHUB_TOKEN` بصلاحية **Contents: Read and write** لهذا المستودع وحده. لا تلصق الرمز في خلية أو ملف أو رسالة ولا تشارك مخرجاته.

رابط Cloudflare المجاني مؤقت ويتغير عند إعادة تشغيل النفق. هذا لا يجعل جلسة Colab دائمة؛ تنتهي الخدمة عندما تتوقف الجلسة أو يُسحب GPU.


In [ ]:
deb = '/content/cloudflared.deb'
subprocess.run([
    'bash', '-lc',
    f'wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O {deb} && dpkg -i {deb}'
], check=True)

cf_log = open('/content/cloudflared.log', 'w')
cf_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8787'],
    stdout=cf_log, stderr=subprocess.STDOUT, text=True
)
tunnel_url = None
for _ in range(60):
    time.sleep(2)
    log_text = Path('/content/cloudflared.log').read_text(errors='ignore')
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log_text)
    if match:
        tunnel_url = match.group(0)
        break
    if cf_proc.poll() is not None:
        break

if not tunnel_url:
    print(Path('/content/cloudflared.log').read_text(errors='ignore')[-4000:])
    raise RuntimeError('Could not obtain a Cloudflare tunnel URL')

print('JASSIM HTTPS URL:', tunnel_url)
print('Public health:', requests.get(tunnel_url + '/health', timeout=20).json())

# Publishing changes dashboard/connection.json on GitHub main.
# Keep this disabled for private tests; enable only when you explicitly want
# Android auto-discovery to use this temporary Colab tunnel.
PUBLISH_ENDPOINT_TO_GITHUB = False

if PUBLISH_ENDPOINT_TO_GITHUB:
    # Publish the changing tunnel address to the stable discovery file on GitHub Pages.
    # Add a GitHub fine-grained token as a Colab Secret named JASSIM_GITHUB_TOKEN
    # with Contents: Read and write access to this repository.
    from google.colab import userdata
    import base64
    GH_TOKEN = userdata.get('JASSIM_GITHUB_TOKEN')
    if not GH_TOKEN:
        raise RuntimeError('Add the Colab Secret JASSIM_GITHUB_TOKEN (repository Contents: Read and write), then rerun this cell.')

    GITHUB_REPO = 'stevennpllk-crypto/-JASSIM-AI-Android'
    REGISTRY_PATH = 'dashboard/connection.json'
    REGISTRY_API = f'https://api.github.com/repos/{GITHUB_REPO}/contents/{REGISTRY_PATH}'
    GH_HEADERS = {
        'Authorization': 'Bearer ' + GH_TOKEN,
        'Accept': 'application/vnd.github+json',
        'X-GitHub-Api-Version': '2022-11-28'
    }

    def _publish_endpoint(url):
        payload = {
            'status': 'online',
            'url': url,
            'updated_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
            'service': 'JASSIM AI'
        }
        current = requests.get(REGISTRY_API, headers=GH_HEADERS, params={'ref': 'main'}, timeout=20)
        sha = current.json().get('sha') if current.status_code == 200 else None
        encoded = base64.b64encode(json.dumps(payload, ensure_ascii=False).encode('utf-8')).decode('ascii')
        commit = {
            'message': 'Update live JASSIM AI endpoint',
            'content': encoded,
            'branch': 'main'
        }
        if sha:
            commit['sha'] = sha
        response = requests.put(REGISTRY_API, headers=GH_HEADERS, json=commit, timeout=30)
        response.raise_for_status()
        print('Published current endpoint to the stable discovery registry.')

    _publish_endpoint(tunnel_url)

else:
    print('Endpoint was NOT published to GitHub. Use the HTTPS URL above for a manual private test.')


## 7. ربط التطبيق والموقع

In [ ]:
print('API URL:', tunnel_url)
print('Database folder in Drive:', DATA_DIR)
if globals().get('PUBLISH_ENDPOINT_TO_GITHUB', False):
    print('Android auto-discovery is enabled by explicit opt-in. Keep the GitHub token private.')
else:
    print('Android auto-discovery is disabled. Copy this HTTPS URL into Android Settings for a private test; keep the API token private.')


## 8. مراقبة العمليات وإعادة تشغيلها تلقائياً (Watchdog)

شغّل الخلية التالية مرة واحدة بعد نجاح الخطوات السابقة. إذا تعطّل Ollama أو API أو النفق، تحاول الخلية إعادة تشغيله. إذا تغيّر رابط النفق ستطبعه من جديد ويجب تحديثه في التطبيق. لا ينشر Watchdog الرابط إلى GitHub تلقائياً.

هذه الميزة لا تمنع Google Colab من إيقاف الجلسة أو سحب GPU. للاستمرارية الدائمة استخدم خادماً يعمل طوال الوقت وتعليمات DEPLOYMENT.md.

In [ ]:
# Watchdog works only while the Colab runtime is alive.
# A None process handle means the service was already running before this notebook
# started; verify it over HTTP instead of calling .poll() on None.
watchdog_stop = threading.Event()
watchdog_state = {'api_failures': 0, 'tunnel_failures': 0}

def _service_reachable(url, timeout=3):
    try:
        response = requests.get(url, timeout=timeout)
        return response.ok
    except Exception:
        return False

def _process_exited(proc):
    return proc is not None and proc.poll() is not None

def _wait_ollama(timeout=90):
    deadline = time.time() + timeout
    while time.time() < deadline:
        if _service_reachable('http://127.0.0.1:11434/api/tags'):
            return True
        time.sleep(3)
    return False

def _start_api():
    global api_proc, api_log
    api_log = open('/content/jassim-api.log', 'a')
    api_proc = subprocess.Popen(
        ['python', str(SERVER)], cwd=str(PROJECT / 'backend'),
        env=os.environ.copy(), stdout=api_log, stderr=subprocess.STDOUT
    )

def _start_tunnel():
    global cf_proc, cf_log, tunnel_url
    cf_log = open('/content/cloudflared.log', 'w')
    cf_proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8787'],
        stdout=cf_log, stderr=subprocess.STDOUT, text=True
    )
    new_url = None
    for _ in range(30):
        time.sleep(2)
        log_text = Path('/content/cloudflared.log').read_text(errors='ignore')
        matches = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', log_text)
        if matches:
            new_url = matches[-1]
            break
        if _process_exited(cf_proc):
            break
    if new_url:
        tunnel_url = new_url
        if globals().get('PUBLISH_ENDPOINT_TO_GITHUB', False) and callable(globals().get('_publish_endpoint')):
            try:
                _publish_endpoint(tunnel_url)
                print('[watchdog] Endpoint published to GitHub by explicit opt-in.')
            except Exception as exc:
                print('[watchdog] Could not publish discovery endpoint:', type(exc).__name__)
        else:
            print('[watchdog] New HTTPS URL is private/manual only; update the app manually if needed.')
    else:
        print('[watchdog] Tunnel did not recover; inspect /content/cloudflared.log')

def _stop_process(proc):
    if proc is None or proc.poll() is not None:
        return
    proc.terminate()
    try:
        proc.wait(timeout=5)
    except subprocess.TimeoutExpired:
        proc.kill()

def _watch_services():
    global ollama_proc, ollama_log, api_proc, cf_proc
    while not watchdog_stop.wait(25):
        try:
            # Ollama may have been started outside this notebook, leaving ollama_proc=None.
            ollama_down = _process_exited(ollama_proc) or not _service_reachable('http://127.0.0.1:11434/api/tags')
            if ollama_down:
                print('[watchdog] Ollama is unavailable; starting/restarting it...')
                if ollama_proc is not None:
                    _stop_process(ollama_proc)
                ollama_log = open('/content/ollama.log', 'a')
                ollama_proc = subprocess.Popen(['ollama', 'serve'], stdout=ollama_log, stderr=subprocess.STDOUT)
                if not _wait_ollama():
                    print('[watchdog] Ollama did not recover; inspect /content/ollama.log')
                    continue

            if api_proc is None or _process_exited(api_proc):
                print('[watchdog] Starting/restarting JASSIM API...')
                _start_api()
                time.sleep(3)

            try:
                health = requests.get('http://127.0.0.1:8787/health', timeout=6)
                health.raise_for_status()
                readiness = requests.get('http://127.0.0.1:8787/ready', timeout=6)
                if readiness.ok and readiness.json().get('ready'):
                    watchdog_state['api_failures'] = 0
                else:
                    print('[watchdog] API is alive but private chat is NOT ready:', readiness.text[:500])
            except Exception as exc:
                print('[watchdog] API health check failed:', type(exc).__name__)
                watchdog_state['api_failures'] += 1
                if watchdog_state['api_failures'] >= 3:
                    print('[watchdog] API readiness failed repeatedly; restarting API...')
                    _stop_process(api_proc)
                    _start_api()
                    watchdog_state['api_failures'] = 0

            tunnel_down = _process_exited(cf_proc) or not tunnel_url or not _service_reachable(tunnel_url + '/health', timeout=8)
            if tunnel_down:
                print('[watchdog] Tunnel is unavailable; starting/restarting Cloudflare tunnel...')
                if cf_proc is not None:
                    _stop_process(cf_proc)
                _start_tunnel()
            else:
                try:
                    tunnel_readiness = requests.get(tunnel_url + '/ready', timeout=8)
                    if not (tunnel_readiness.ok and tunnel_readiness.json().get('ready')):
                        print('[watchdog] Public API is reachable, but private model readiness is false:', tunnel_readiness.text[:500])
                    watchdog_state['tunnel_failures'] = 0
                except Exception:
                    watchdog_state['tunnel_failures'] += 1
                    if watchdog_state['tunnel_failures'] >= 3:
                        print('[watchdog] Public tunnel failed repeatedly; replacing it...')
                        _stop_process(cf_proc)
                        _start_tunnel()
                        watchdog_state['tunnel_failures'] = 0
        except Exception as exc:
            print('[watchdog] Monitor error:', type(exc).__name__, str(exc)[:300])

watchdog_thread = threading.Thread(target=_watch_services, daemon=True)
watchdog_thread.start()
print('Watchdog active. Stop it with watchdog_stop.set()')


## 9. إظهار رمز الربط عند الحاجة فقط
لا تشغّل هذه الخلية إلا عندما تريد إدخال الرمز يدوياً في تطبيق Android. لا تشارك لقطة الشاشة أو مخرجات هذه الخلية، ولا تحفظ نسخة عامة من الدفتر بعد تشغيلها.

In [ ]:
# The API token is stored in Google Drive with restricted file permissions.
# Do not reveal it in notebook output unless you are intentionally configuring a trusted client.
SHOW_API_TOKEN = False
if SHOW_API_TOKEN:
    from IPython.display import display, Markdown
    display(Markdown('**رمز API الخاص — لا تشاركه:**'))
    print(API_TOKEN)
else:
    print('API token remains hidden and stored in Google Drive.')
